[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.5_agentic_workload/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.5_agentic_workload/lab.ipynb)

# 10.5 Lab: Agentic Workload Infrastructure Simulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.5_agentic_workload/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.5_agentic_workload/lab.ipynb)

Simulate 10K concurrent agent sessions with multi-step execution, KV cache growth, tiered memory, and session-aware routing. Measure cache hit rates, memory pressure, and cost under the two-model architecture.


In [ ]:
# Install dependencies via subprocess for Colab/Molab compatibility
import subprocess
import sys
# numpy: random sampling and array operations for session simulation
# matplotlib: visualize KV cache growth, memory tiers, and cost distribution
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib"])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === PARAMETERS (change these and re-run) ===
# Number of concurrent agent sessions to simulate
NUM_SESSIONS = 10000
# Random seed for reproducible experiments
SEED = 42
# Maximum steps an agent can take before terminating
MAX_STEPS = 20

# KV cache constants for a 70B model with GQA (8 KV heads)
# Formula: 2(K+V) x 8(heads) x 128(dim) x 80(layers) x 2(FP16 bytes) = 327,680 bytes/token
KV_BYTES_PER_TOKEN = 327680
# Convert to GB for readability in charts
KV_GB_PER_TOKEN = KV_BYTES_PER_TOKEN / (1024**3)

# Token accumulation per step (system prompt + tool results + reasoning)
# Based on production traces: starts at 2K, grows ~3-5K per step
BASE_TOKENS = 2000      # initial context (system prompt + user query)
TOKENS_PER_STEP = 4000  # average tokens added per agent step (tool output + reasoning)

# Memory tier capacities (per GPU node)
GPU_HBM_GB = 64.0      # available after model weights loaded (H100 80GB - 16GB weights)
HOST_DRAM_GB = 512.0   # host memory for warm sessions (standard server RAM)

# Two-model architecture cost parameters (per 1K output tokens)
COST_70B_PER_1K = 0.004   # planning/reasoning model cost
COST_8B_PER_1K = 0.0008   # execution/formatting model cost
# Fraction of steps that go to 70B (planning steps: plan, reason, recover, synthesize)
PLANNING_STEP_FRACTION = 0.4


In [ ]:
# === SIMULATION: Generate 10K agent sessions with varying depths ===
# Initialize reproducible random number generator
rng = np.random.default_rng(SEED)

# Sample session depths from a realistic distribution
# Most sessions are 5-7 steps; some go deep (15-20 for complex tasks)
# Using a clipped normal distribution centered at 7 steps
raw_depths = rng.normal(loc=7, scale=3, size=NUM_SESSIONS)
# Clip to valid range: minimum 2 steps, maximum MAX_STEPS
session_depths = np.clip(raw_depths, 2, MAX_STEPS).astype(int)

# Calculate KV cache size at each session's current depth
# Total tokens = base context + (depth x tokens_per_step)
session_tokens = BASE_TOKENS + session_depths * TOKENS_PER_STEP
# Convert token count to KV cache size in GB
session_kv_gb = session_tokens * KV_GB_PER_TOKEN

# Report aggregate memory requirements
total_kv_tb = session_kv_gb.sum() / 1024  # convert GB to TB
print(f"=== Agentic Workload Memory Analysis ({NUM_SESSIONS:,} sessions) ===")
print(f"\nSession depth stats:")
print(f"  Mean: {session_depths.mean():.1f} steps")
print(f"  Median: {np.median(session_depths):.0f} steps")
print(f"  P95: {np.percentile(session_depths, 95):.0f} steps")
print(f"  P99: {np.percentile(session_depths, 99):.0f} steps")
print(f"\nKV cache requirements:")
print(f"  Per session (mean): {session_kv_gb.mean():.2f} GB")
print(f"  Per session (P95): {np.percentile(session_kv_gb, 95):.2f} GB")
print(f"  Aggregate total: {total_kv_tb:.1f} TB")
print(f"  This exceeds GPU memory by {total_kv_tb / (GPU_HBM_GB/1024):.0f}x --> tiered storage required")


In [ ]:
# === VISUALIZATION 1: KV cache growth curve across agent steps ===
fig_c3, axes_c3 = plt.subplots(1, 2, figsize=(13, 5))

# Left panel: KV cache size vs step depth (single session perspective)
# Generate step numbers from 1 to MAX_STEPS for x-axis
steps_range = np.arange(1, MAX_STEPS + 1)
# Calculate KV cache at each step for one session
# Total context tokens grows linearly with each agent step
tokens_at_step = BASE_TOKENS + steps_range * TOKENS_PER_STEP
# Convert to GB using the per-token KV size
# Convert token count to KV cache size in GB
kv_at_step_gb = tokens_at_step * KV_GB_PER_TOKEN

# Plot the monotonic growth curve that makes agents so hard to serve
axes_c3[0].plot(steps_range, kv_at_step_gb, color="#2563eb", linewidth=2.5, marker="o", markersize=4)
# Add horizontal line showing single-GPU HBM capacity
axes_c3[0].axhline(y=GPU_HBM_GB, color="#991b1b", linestyle="--", linewidth=1.5,
                label=f"GPU HBM capacity ({GPU_HBM_GB} GB)")
# Shade the region beyond GPU memory in red (requires spilling)
axes_c3[0].fill_between(steps_range, GPU_HBM_GB, kv_at_step_gb,
                     where=kv_at_step_gb > GPU_HBM_GB, alpha=0.2, color="#ffe4e6",
                     label="Spills to DRAM/distributed")
axes_c3[0].set_xlabel("Agent Step", fontsize=11)
axes_c3[0].set_ylabel("KV Cache Size (GB)", fontsize=11)
axes_c3[0].set_title("KV Cache Growth Per Session", fontsize=13, fontweight="bold")
axes_c3[0].legend(fontsize=9)
axes_c3[0].grid(True, alpha=0.3)

# Right panel: histogram of session depths across all 10K sessions
axes_c3[1].hist(session_depths, bins=range(2, MAX_STEPS + 2), color="#dbeafe",
             edgecolor="#000", linewidth=0.8, alpha=0.9)
# Mark the mean depth with a vertical line
axes_c3[1].axvline(x=session_depths.mean(), color="#2563eb", linestyle="--",
                linewidth=1.5, label=f"Mean: {session_depths.mean():.1f} steps")
axes_c3[1].set_xlabel("Session Depth (steps)", fontsize=11)
axes_c3[1].set_ylabel("Number of Sessions", fontsize=11)
axes_c3[1].set_title(f"Session Depth Distribution (N={NUM_SESSIONS:,})", fontsize=13, fontweight="bold")
axes_c3[1].legend(fontsize=10)
axes_c3[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("kv_cache_growth.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: kv_cache_growth.png")


In [ ]:
# === EXPERIMENT 2: Tiered memory allocation simulation ===
# Classify each session into memory tiers based on recency and size
# Tier 1 (GPU HBM): active sessions that fit in GPU memory
# Tier 2 (Host DRAM): warm sessions evicted from GPU but kept in host RAM
# Tier 3 (Distributed): cold sessions stored in LMCache/Redis

# Sort sessions by KV size (smallest first, representing most recent/active)
sorted_kv = np.sort(session_kv_gb)
# Cumulative sum tells us how many sessions fit in each tier
cumulative_kv = np.cumsum(sorted_kv)

# How many sessions fit in GPU HBM? (active tier)
# Count sessions where cumulative KV stays below GPU capacity
tier1_count = np.searchsorted(cumulative_kv, GPU_HBM_GB)
# How many additional sessions fit in host DRAM? (warm tier)
tier2_count = np.searchsorted(cumulative_kv, GPU_HBM_GB + HOST_DRAM_GB) - tier1_count
# Remaining sessions go to distributed cache (cold tier)
tier3_count = NUM_SESSIONS - tier1_count - tier2_count

# Compute cache hit rate: sessions served from GPU without any transfer latency
# In practice, session-aware routing keeps ~95% in Tier 1
# Here we calculate the theoretical lower bound (random placement)
cache_hit_rate = tier1_count / NUM_SESSIONS * 100

print(f"=== Tiered Memory Allocation ===")
print(f"\nTier 1 (GPU HBM, {GPU_HBM_GB} GB): {tier1_count:,} sessions ({tier1_count/NUM_SESSIONS*100:.1f}%)")
print(f"Tier 2 (Host DRAM, {HOST_DRAM_GB} GB): {tier2_count:,} sessions ({tier2_count/NUM_SESSIONS*100:.1f}%)")
print(f"Tier 3 (Distributed cache): {tier3_count:,} sessions ({tier3_count/NUM_SESSIONS*100:.1f}%)")
print(f"\nTheoretical cache hit rate (random): {cache_hit_rate:.1f}%")
print(f"With session-aware routing (production): ~95% (affinity keeps hot sessions on-GPU)")


In [ ]:
# === VISUALIZATION 3: Memory tier breakdown (stacked bar) ===
fig_c5, axes_c5 = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: stacked bar showing tier allocation
# Categories represent the three memory tiers
# Count of sessions in each memory tier
tier_sizes = [tier1_count, tier2_count, tier3_count]
tier_labels = [f"GPU HBM\n({tier1_count:,})", f"Host DRAM\n({tier2_count:,})", f"Distributed\n({tier3_count:,})"]
tier_colors = ["#dbeafe", "#fef3c7", "#f3f4f6"]

# Draw stacked horizontal bar (proportion of sessions per tier)
left = 0
for size, label, color in zip(tier_sizes, tier_labels, tier_colors):
    # Width proportional to fraction of total sessions
    width = size / NUM_SESSIONS
    axes_c5[0].barh(0, width, left=left, color=color, edgecolor="#000", linewidth=1.2,
                 label=label, height=0.5)
    # Label each segment with session count
    if width > 0.05:
        axes_c5[0].text(left + width/2, 0, f"{size:,}", ha="center", va="center", fontsize=10)
    left += width

axes_c5[0].set_xlim(0, 1)
axes_c5[0].set_xlabel("Fraction of Sessions", fontsize=11)
axes_c5[0].set_title("Memory Tier Distribution", fontsize=13, fontweight="bold")
axes_c5[0].legend(loc="upper right", fontsize=9)
axes_c5[0].set_yticks([])

# Right panel: cost comparison between two-model vs single-model architecture
# Calculate total tokens generated across all sessions (output tokens)
# Average 200 output tokens per step
# Estimate output tokens: 200 tokens generated per agent step
output_tokens_per_session = session_depths * 200
# Aggregate all output tokens across all sessions
total_output_tokens = output_tokens_per_session.sum()

# Two-model cost: 40% steps to 70B, 60% steps to 8B
# Cost from planning steps routed to 70B model
cost_70b_share = total_output_tokens * PLANNING_STEP_FRACTION * COST_70B_PER_1K / 1000
# Cost from execution steps routed to 8B model
cost_8b_share = total_output_tokens * (1 - PLANNING_STEP_FRACTION) * COST_8B_PER_1K / 1000
cost_two_model = cost_70b_share + cost_8b_share

# Single-model cost: all steps to 70B
cost_single_model = total_output_tokens * COST_70B_PER_1K / 1000

# Bar chart comparing the two strategies
strategies = ["Two-Model\n(70B + 8B)", "Single-Model\n(all 70B)"]
costs = [cost_two_model, cost_single_model]
bar_colors = ["#dcfce7", "#ffe4e6"]
bars = axes_c5[1].bar(strategies, costs, color=bar_colors, edgecolor="#000", linewidth=1.2)
axes_c5[1].set_ylabel("Total Inference Cost ($)", fontsize=11)
axes_c5[1].set_title("Two-Model Architecture Savings", fontsize=13, fontweight="bold")
# Annotate exact dollar amounts on bars
for bar, cost in zip(bars, costs):
    axes_c5[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f"${cost:.2f}", ha="center", fontsize=11, fontweight="bold")
# Show savings percentage as annotation
# Percentage cost reduction from two-model architecture
savings_pct = (1 - cost_two_model / cost_single_model) * 100
axes_c5[1].annotate(f"Saves {savings_pct:.0f}%", xy=(0, cost_two_model),
                xytext=(0.5, (cost_two_model + cost_single_model)/2),
                fontsize=12, color="#166534", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#166534"))

plt.tight_layout()
plt.savefig("memory_tiers_and_cost.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: memory_tiers_and_cost.png")
print(f"\nTwo-model savings: {savings_pct:.1f}% (${cost_single_model - cost_two_model:.2f} saved)")


In [ ]:
# === EXPERIMENT 4: Session-aware routing simulation ===
# Simulate the scheduler's affinity map: sessions pinned to GPUs
# Model: 100 GPU slots, sessions arrive and depart

NUM_GPUS = 100          # number of GPU slots in the cluster
SIM_DURATION_S = 300    # simulate 5 minutes of traffic
# Each session makes one step every 3 seconds on average
STEP_INTERVAL_S = 3.0

# Track routing decisions
cache_hits = 0     # session found on its pinned GPU (0ms overhead)
cache_misses = 0   # session evicted, needs reload (5-50ms overhead)

# Affinity map: session_id -> gpu_id
affinity = {}
# GPU occupancy: gpu_id -> set of session_ids currently loaded
gpu_sessions = {i: set() for i in range(NUM_GPUS)}
# Max sessions per GPU based on memory (GPU_HBM / avg_session_kv)
avg_kv_gb = session_kv_gb.mean()
max_per_gpu = max(1, int(GPU_HBM_GB / avg_kv_gb))

# Generate step events: (time, session_id)
# Each session generates steps at random intervals around STEP_INTERVAL_S
events = []
for sid in range(NUM_SESSIONS):
    # Each session starts at a random time within the simulation window
    start = rng.uniform(0, SIM_DURATION_S * 0.5)
    # Generate steps for this session's full depth
    for step in range(session_depths[sid]):
        # Time of this step: start + step * interval + jitter
        t = start + step * STEP_INTERVAL_S + rng.exponential(0.5)
        if t < SIM_DURATION_S:
            events.append((t, sid))

# Sort events by time (chronological processing)
events.sort(key=lambda x: x[0])

# Process each step event through the affinity router
for t, sid in events:
    if sid in affinity:
        # Session has a pinned GPU -- check if still loaded
        gpu_id = affinity[sid]
        if sid in gpu_sessions[gpu_id]:
            # Cache hit: session KV is already on this GPU
            cache_hits += 1
        else:
            # Cache miss: session was evicted, need to reload from DRAM
            cache_misses += 1
            # Reload onto the same GPU (maintain affinity)
            if len(gpu_sessions[gpu_id]) >= max_per_gpu:
                # GPU full: evict oldest session (LRU policy)
                evicted = gpu_sessions[gpu_id].pop()
                del affinity[evicted]
            gpu_sessions[gpu_id].add(sid)
    else:
        # New session: assign to least-loaded GPU
        cache_misses += 1
        # Find GPU with fewest sessions (load balancing)
        least_loaded = min(gpu_sessions, key=lambda g: len(gpu_sessions[g]))
        # If GPU is full, evict one session to make room
        if len(gpu_sessions[least_loaded]) >= max_per_gpu:
            evicted = gpu_sessions[least_loaded].pop()
            if evicted in affinity:
                del affinity[evicted]
        # Pin this session to the chosen GPU
        affinity[sid] = least_loaded
        gpu_sessions[least_loaded].add(sid)

# Report routing efficiency
total_events = cache_hits + cache_misses
hit_rate = cache_hits / total_events * 100
print(f"=== Session-Aware Routing Results ===")
print(f"Total step events processed: {total_events:,}")
print(f"Cache hits (0ms overhead):   {cache_hits:,} ({hit_rate:.1f}%)")
print(f"Cache misses (5-50ms reload): {cache_misses:,} ({100-hit_rate:.1f}%)")
print(f"\nGPU slots: {NUM_GPUS}, Max sessions/GPU: {max_per_gpu}")
print(f"Effective cluster capacity: {NUM_GPUS * max_per_gpu:,} concurrent sessions")
